# EBRAINS BrainScaleS-2 TTFS neuron-pooling experiment

Use the `EBRAINS-experimental` kernel and execute cells in order. This notebook is a thin launcher for the repository CLI. It deliberately does **not** install the full project `requirements.txt`, upgrade `torch`, or install `hxtorch`.

## 1. Install only notebook-side dependencies

`%pip` targets the active Jupyter kernel. The BrainScaleS-2 software stack remains the version supplied by EBRAINS.

In [ ]:
%pip install --quiet --disable-pip-version-check jaxtyping matplotlib

## 2. Locate the repository and select run stages

CADC diagnosis, a small operating-point sweep, and hardware smoke are enabled by default; `Run All` therefore requests shared hardware. The full experiment remains disabled.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import csv
import json
import os
import subprocess
import sys

start = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in (start, *start.parents)
        if (candidate / "scripts/evaluation/brainscales2_pooling.py").is_file()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Could not locate the delayed-temporal repository root")
os.chdir(repo_root)

RUN_MOCK_SMOKE = True
RUN_CADC_DIAGNOSTIC = True
RUN_HARDWARE_SMOKE = True
RUN_OPERATING_POINT_SWEEP = True
RUN_FULL_EXPERIMENT = False

# Formal full runs require an explicit, immutable .pbin calibration.
CALIBRATION_PATH = None  # Example: Path("/mnt/user/shared/calibration/spiking_calibration.pbin")
DEMOS_ROOT = Path("/tmp/brainscales2-demos")
RUN_LABEL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ARTIFACT_ROOT = repo_root / "artifacts/brainscales2" / RUN_LABEL
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
SMOKE_CALIBRATION_PATH = ARTIFACT_ROOT / "spiking_cocolist.pbin"

print("repository:", repo_root)
print("python:", sys.executable)
print("artifacts:", ARTIFACT_ROOT)

## 3. Check the active EBRAINS kernel

In [ ]:
import inspect
import torch
import jaxtyping
import hxtorch
import hxtorch.spiking as hxsnn

print("Python:", sys.version)
print("torch:", torch.__version__)
print("hxtorch:", getattr(hxtorch, "__version__", "unknown"))
print("Experiment:", inspect.signature(hxsnn.Experiment))
print("Synapse:", inspect.signature(hxsnn.Synapse))
print("LIF:", inspect.signature(hxsnn.LIF))
print("run:", inspect.signature(hxsnn.run))

## 4. Configure the EBRAINS hardware client

When any hardware stage is enabled, this cell configures the client and saves the official nightly calibration into the run directory. Smoke stages load that `.pbin` explicitly, avoiding an implicit full-chip calibration.

In [ ]:
hardware_requested = any(
    (
        RUN_HARDWARE_SMOKE,
        RUN_CADC_DIAGNOSTIC,
        RUN_OPERATING_POINT_SWEEP,
        RUN_FULL_EXPERIMENT,
    )
)

if hardware_requested:
    if not DEMOS_ROOT.is_dir():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "jupyter-notebooks-experimental",
                "https://github.com/electronicvisions/brainscales2-demos.git",
                str(DEMOS_ROOT),
            ],
            check=True,
        )
    if str(DEMOS_ROOT) not in sys.path:
        sys.path.insert(0, str(DEMOS_ROOT))
    from _static.common.helpers import save_nightly_calibration, setup_hardware_client
    setup_hardware_client()
    if not SMOKE_CALIBRATION_PATH.is_file():
        save_nightly_calibration(
            SMOKE_CALIBRATION_PATH.name,
            folder=str(SMOKE_CALIBRATION_PATH.parent),
        )
    if not SMOKE_CALIBRATION_PATH.is_file():
        raise FileNotFoundError(SMOKE_CALIBRATION_PATH)
    print("EBRAINS hardware client configured")
    print("smoke calibration:", SMOKE_CALIBRATION_PATH)
else:
    print("Hardware stages disabled; client setup skipped")

## 5. CLI helpers

Every CLI subprocess uses the active kernel's exact Python executable and inherits the hardware-client environment.

In [ ]:
CLI = repo_root / "scripts/evaluation/brainscales2_pooling.py"
VERIFY = repo_root / "scripts/verification/verify_brainscales2_pooling.py"

def run_python(script: Path, *arguments: object) -> None:
    command = [sys.executable, str(script), *(str(value) for value in arguments)]
    print(" ".join(command))
    subprocess.run(command, cwd=repo_root, check=True)

def show_artifacts(output_dir: Path) -> None:
    manifest_path = output_dir / "manifest.json"
    summary_path = output_dir / "summary.csv"
    if not manifest_path.is_file() or not summary_path.is_file():
        print("No completed artifacts:", output_dir)
        return
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    with summary_path.open(newline="", encoding="utf-8") as handle:
        summary = list(csv.DictReader(handle))
    print(json.dumps(
        {
            "schema_version": manifest.get("schema_version"),
            "conditions": len(manifest.get("conditions", [])),
            "environment": manifest.get("environment"),
        },
        indent=2,
    ))
    display(summary[:8])
    for figure_name in ("cadc_trace.png", "variance_fit.png"):
        figure = output_dir / figure_name
        if figure.is_file():
            from IPython.display import Image
            display(Image(filename=str(figure)))

## 6. Pure-Python verification

This validates encoding, routing shapes, miss-aware pooling, variance-floor recovery, artifacts, and notebook metadata without accessing hardware.

In [ ]:
run_python(VERIFY)

## 7. Mock quick smoke

This runs three potential positions, pool sizes 1 and 4, one placement/routing condition, and eight trials.

In [ ]:
mock_dir = ARTIFACT_ROOT / "mock_quick"
if RUN_MOCK_SMOKE:
    run_python(
        CLI,
        "--backend", "mock",
        "--quick",
        "--output-dir", mock_dir,
    )
    show_artifacts(mock_dir)
else:
    print("Mock smoke disabled")

## 8. CADC single-PSP diagnostic

This follows the official hxtorch SNN introduction: compare no-input and one-input CADC membrane traces before attempting timing measurements. CADC is diagnostic only; final jitter still uses raw spike events.

In [ ]:
cadc_diagnostic_dir = ARTIFACT_ROOT / "cadc_diagnostic"
cadc_recommendation_path = cadc_diagnostic_dir / "recommended_operating_point.json"
if RUN_CADC_DIAGNOSTIC:
    run_python(
        CLI,
        "--phase", "diagnose-cadc",
        "--backend", "hardware",
        "--quick",
        "--calibration", SMOKE_CALIBRATION_PATH,
        "--i-synin-gm", 700,
        "--output-dir", cadc_diagnostic_dir,
    )
    show_artifacts(cadc_diagnostic_dir)
    diagnostic = json.loads(cadc_recommendation_path.read_text(encoding="utf-8"))
    display(diagnostic)
    if not diagnostic["viable"]:
        raise RuntimeError(
            f'{diagnostic["reason"]}; stop before raw-spike sweeps'
        )
else:
    print("CADC diagnostic disabled")

## 9. Raw-spike operating-point sweep

After CADC confirms a physical PSP, this quick sweep tests actual raw firing across threshold and gain settings. It rejects an all-miss or multi-spike selection before the jitter smoke.

In [ ]:
operating_point_dir = ARTIFACT_ROOT / "operating_point"
selected_path = operating_point_dir / "selected_operating_point.json"
if RUN_OPERATING_POINT_SWEEP:
    run_python(
        CLI,
        "--phase", "calibrate",
        "--backend", "hardware",
        "--quick",
        "--calibration", SMOKE_CALIBRATION_PATH,
        "--calibration-thresholds", 100, 110, 125,
        "--calibration-weights", 63,
        "--calibration-gains", 500, 700,
        "--output-dir", operating_point_dir,
    )
    selection = json.loads(selected_path.read_text(encoding="utf-8"))
    display(selection)
    selected = selection["selected"]
    if (
        selected["fired_rate"] < 0.8
        or selected["multi_spike_rate"] > 0.05
        or selected["premature_spike_rate"] > 0.05
    ):
        raise RuntimeError(
            f"No usable raw-spike operating point in the quick grid: {selected}"
        )
else:
    print("Operating-point sweep disabled")

## 10. Hardware quick smoke

This runs only after CADC confirms a measurable PSP and the raw-spike sweep selects a firing operating point. It reuses that selected point instead of the previous all-miss defaults.

In [ ]:
hardware_smoke_dir = ARTIFACT_ROOT / "hardware_smoke"
if RUN_HARDWARE_SMOKE:
    if not selected_path.is_file():
        raise FileNotFoundError(f"Run the operating-point sweep first: {selected_path}")
    run_python(
        CLI,
        "--backend", "hardware",
        "--quick",
        "--calibration", SMOKE_CALIBRATION_PATH,
        "--operating-point-json", selected_path,
        "--output-dir", hardware_smoke_dir,
    )
    show_artifacts(hardware_smoke_dir)
else:
    print("Hardware smoke disabled")

## 11. Formal pooling experiment

The full experiment requires both an explicit calibration `.pbin` and a selected operating point. It runs pool sizes 1, 2, 4, 8, and 16 across both placement and routing conditions with 256 trials.

In [ ]:
full_dir = ARTIFACT_ROOT / "full_pooling"
selected_path = operating_point_dir / "selected_operating_point.json"

if RUN_FULL_EXPERIMENT:
    if CALIBRATION_PATH is None:
        raise RuntimeError("Set CALIBRATION_PATH to an explicit .pbin before a formal run")
    calibration_path = Path(CALIBRATION_PATH).expanduser().resolve()
    if not calibration_path.is_file():
        raise FileNotFoundError(calibration_path)
    if not selected_path.is_file():
        raise FileNotFoundError(
            f"Run the operating-point sweep first: {selected_path}"
        )
    run_python(
        CLI,
        "--backend", "hardware",
        "--encoding", "identity",
        "--pool-sizes", 1, 2, 4, 8, 16,
        "--placements", "same-quadrant", "cross-quadrant",
        "--routing", "broadcast", "independent",
        "--trials", 256,
        "--calibration", calibration_path,
        "--operating-point-json", selected_path,
        "--output-dir", full_dir,
    )
    show_artifacts(full_dir)
else:
    print("Full experiment disabled")

## 12. Result locations

Download or persist the complete run directory, including `manifest.json`, raw `events.csv`, `events.pt`, summaries, variance fit, and figure.

In [ ]:
for path in sorted(ARTIFACT_ROOT.glob("*")):
    print(path)